# HW2 - Modifications & Modkit

Theodore M. Nelson

The teaching objective is to learn the modkit package and introduce modification basecalling in Nanopore sequencing data.

## Download HCV Data

These are in-vitro transcribed HCV genome signals (https://www.mayoclinic.org/diseases-conditions/hepatitis-c/symptoms-causes/syc-20354278).

In [2]:
# Step 2: Download the FAST5 file
! wget -O HCV_IVT_004_500_RANDOM_READS.fast5 "https://zenodo.org/records/10989179/files/HCV_IVT_004_500_RANDOM_READS.fast5?download=1"

--2025-04-12 14:20:57--  https://zenodo.org/records/10989179/files/HCV_IVT_004_500_RANDOM_READS.fast5?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.194, 188.185.43.25, 188.185.45.92, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 108570398 (104M) [application/octet-stream]
Saving to: ‘HCV_IVT_004_500_RANDOM_READS.fast5’

HCV_IVT_004_500_RAN 100%[===================>] 103.54M  11.1MB/s    in 11s     

2025-04-12 14:21:08 (9.70 MB/s) - ‘HCV_IVT_004_500_RANDOM_READS.fast5’ saved [108570398/108570398]



These are HCV signals from a Huh7-infected cell culture system, selected based on known alignment to the HCV genome.

In [1]:
! wget https://github.com/Theo-Nelson/SMS-data/raw/refs/heads/main/HCV/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5

--2025-04-12 14:20:15--  https://github.com/Theo-Nelson/SMS-data/raw/refs/heads/main/HCV/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Theo-Nelson/SMS-data/refs/heads/main/HCV/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5 [following]
--2025-04-12 14:20:16--  https://raw.githubusercontent.com/Theo-Nelson/SMS-data/refs/heads/main/HCV/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8541977 (8.1M) [application/octet-stream]
Saving to: ‘HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5’

HCV_TotalRNA_004_AL 100%[===================>]   8.1

## Convert Blow5 to Pod5

We next convert the blow5 file containing the HCV reads sourced from total RNA to the pod5 format, in advance of basecalling.

In [3]:
# Install system dependencies
!apt-get update
!apt-get install -y python3.8 python3.8-venv python3.8-dev zlib1g-dev

# Create and activate virtualenv
!python3.8 -m venv blue-crab-venv
!source blue-crab-venv/bin/activate && pip install --upgrade pip && pip install blue-crab

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,688 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,824 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Get:

In [4]:
# Run conversion in the virtual environment
! source blue-crab-venv/bin/activate && \
  blue-crab s2p /content/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5 -o /content/HCV_TotalRNA_004_ALIGNED_SIGNALS.pod5

12-Apr-25 14:26:43 - blue-crab - [INFO]: single2single: 1 s/blow5 file detected as input. Writing 1:1 s/blow5->pod5 to file: /content/HCV_TotalRNA_004_ALIGNED_SIGNALS.pod5
12-Apr-25 14:26:43 - blue-crab - [INFO]: Opening s/blow5 file: /content/HCV_TotalRNA_004_ALIGNED_SIGNALS.blow5
12-Apr-25 14:26:43 - pyslow5 - [WARNING]: get_header_value header value not found: sequencer_hardware_revision - rg: 0
12-Apr-25 14:26:43 - pyslow5 - [WARNING]: get_header_value header value not found: sequencer_serial_number - rg: 0
12-Apr-25 14:26:44 - blue-crab - [INFO]: s/blow5 -> pod5 complete


## Basecalling with Dorado

We need to install dorado first, in this case version 0.8.0 (https://github.com/nanoporetech/dorado/releases).

In [5]:
# Download Dorado v0.8.0
!wget https://cdn.oxfordnanoportal.com/software/analysis/dorado-0.8.0-linux-x64.tar.gz
!tar -xzvf dorado-0.8.0-linux-x64.tar.gz
!chmod +x dorado-0.8.0-linux-x64/bin/dorado

# Set environment variables
DORADO="./dorado-0.8.0-linux-x64/bin/dorado"
BASE_MODEL="rna004_130bps_sup@v5.1.0"


--2025-04-12 14:28:05--  https://cdn.oxfordnanoportal.com/software/analysis/dorado-0.8.0-linux-x64.tar.gz
Resolving cdn.oxfordnanoportal.com (cdn.oxfordnanoportal.com)... 108.138.94.31, 108.138.94.96, 108.138.94.29, ...
Connecting to cdn.oxfordnanoportal.com (cdn.oxfordnanoportal.com)|108.138.94.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2593158106 (2.4G) [application/x-tar]
Saving to: ‘dorado-0.8.0-linux-x64.tar.gz’

dorado-0.8.0-linux- 100%[===================>]   2.42G  28.5MB/s    in 94s     

2025-04-12 14:29:40 (26.3 MB/s) - ‘dorado-0.8.0-linux-x64.tar.gz’ saved [2593158106/2593158106]

dorado-0.8.0-linux-x64/bin/
dorado-0.8.0-linux-x64/bin/dorado
dorado-0.8.0-linux-x64/lib/
dorado-0.8.0-linux-x64/lib/libnvrtc.so.11.2
dorado-0.8.0-linux-x64/lib/libdorado_torch_lib.so
dorado-0.8.0-linux-x64/lib/libnvToolsExt.so.1
dorado-0.8.0-linux-x64/lib/libsz.so.2.0.1
dorado-0.8.0-linux-x64/lib/libnvrtc.so.11.8.89
dorado-0.8.0-linux-x64/lib/libhdf5.so.8
dorado-

We then need to download the kmer models that we wish to use.

In [7]:
# Download the main basecalling model
!/content/dorado-0.8.0-linux-x64/bin/dorado download --model rna004_130bps_sup@v5.1.0

# Download the modification models
!/content/dorado-0.8.0-linux-x64/bin/dorado download --model rna004_130bps_sup@v5.1.0_inosine_m6A@v1
!/content/dorado-0.8.0-linux-x64/bin/dorado download --model rna004_130bps_sup@v5.1.0_pseU@v1
!/content/dorado-0.8.0-linux-x64/bin/dorado download --model rna004_130bps_sup@v5.1.0_m5C@v1

[2025-04-12 14:33:45.851] [info] Running: "download" "--model" "rna004_130bps_sup@v5.1.0"
[2025-04-12 14:33:45.855] [info]  - downloading rna004_130bps_sup@v5.1.0 with httplib
[2025-04-12 14:33:56.243] [info] Running: "download" "--model" "rna004_130bps_sup@v5.1.0_inosine_m6A@v1"
[2025-04-12 14:33:56.246] [info]  - downloading rna004_130bps_sup@v5.1.0_inosine_m6A@v1 with httplib
[2025-04-12 14:33:58.217] [info] Running: "download" "--model" "rna004_130bps_sup@v5.1.0_pseU@v1"
[2025-04-12 14:33:58.220] [info]  - downloading rna004_130bps_sup@v5.1.0_pseU@v1 with httplib
[2025-04-12 14:33:59.918] [info] Running: "download" "--model" "rna004_130bps_sup@v5.1.0_m5C@v1"
[2025-04-12 14:33:59.922] [info]  - downloading rna004_130bps_sup@v5.1.0_m5C@v1 with httplib


The following commands perform basecalling for the HCV Total RNA sample.

In [16]:
DORADO_V080="/content/dorado-0.8.0-linux-x64/bin/dorado"
BASE_MODEL="/content/rna004_130bps_sup@v5.1.0"
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_inosine_m6A@v1"
INPUT="/content/HCV_TotalRNA_004_ALIGNED_SIGNALS.pod5"
OUTPUT="/content/HCV_TotalRNA_inosine_m6A.bam"
LOG="/content/HCV_TotalRNA_inosine_m6A.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

In [17]:
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_pseU@v1"
OUTPUT="/content/HCV_TotalRNA_pseU.bam"
LOG="/content/HCV_TotalRNA_pseU.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

In [18]:
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_m5C@v1"
OUTPUT="/content/HCV_TotalRNA_m5C.bam"
LOG="/content/HCV_TotalRNA_m5C.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

## Questions

1.] What metric tells you the difference in basecalling speed for a pod5 file? How is it defined? What values did you observe above?

**ANSWER HERE**

2.] How are the `--modified-bases-threshold` and `--estimate-poly-a` parameters defined?

**ANSWER HERE**

3.] The following three commands perform basecalling on fast5 files. As this is a deprecated format, the basecaller performs very poorly in runtime. Convert these to pod5 format (either by converting fast5 to blow5 to pod5, or fast5 to pod5 directly) and modify these commands to perform modification basecalling on the IVT samples.

In [ ]:
DORADO_V080="/content/dorado-0.8.0-linux-x64/bin/dorado"
BASE_MODEL="/content/rna004_130bps_sup@v5.1.0"
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_inosine_m6A@v1"
INPUT="/content/HCV_IVT_004_500_RANDOM_READS.fast5"
OUTPUT="/content/HCV_IVT_004_500_RANDOM_READS_inosine_m6A.bam"
LOG="/content/HCV_IVT_004_500_RANDOM_READS_inosine_m6A.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

In [ ]:
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_pseU@v1"
OUTPUT="/content/HCV_IVT_004_500_RANDOM_READS_pseU.bam"
LOG="/content/HCV_IVT_004_500_RANDOM_READS_pseU.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

In [ ]:
MOD_MODEL="/content/rna004_130bps_sup@v5.1.0_m5C@v1"
OUTPUT="/content/HCV_IVT_004_500_RANDOM_READS_m5C.bam"
LOG="/content/HCV_IVT_004_500_RANDOM_READS_m5C.log"

!$DORADO_V080 basecaller $BASE_MODEL "$INPUT" \
  --estimate-poly-a \
  --modified-bases-threshold 0.0 \
  --modified-bases-models $MOD_MODEL \
  > "$OUTPUT" 2> "$LOG"

4.] Why do we need an in-vitro transcribed control for modification calling?

**ANSWER HERE**

5.] Download and run the modkit package from Oxford Nanopore Technologies (https://github.com/nanoporetech/modkit). Answer your own biological question that compares modification calls between IVT and Total HCV reads and uses at least two modkit subcommands (https://github.com/nanoporetech/modkit/blob/master/book/src/advanced_usage.md).

In [ ]:
! wget https://github.com/nanoporetech/modkit/releases/download/v0.4.4/modkit_v0.4.4_u16_x86_64.tar.gz